# Automatisation DPD

Ce notebook montre comment structurer le code pour automatiser la facturation DPD et générer un fichier Python correspondant.

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd

print('Notebook ready')

: 

## 1. Importer les bibliothèques nécessaires

Importer les bibliothèques Python nécessaires pour la logique du notebook et l'automatisation.

In [ ]:
# Section 1: importer les bibliothèques nécessaires
import os
import re
import csv
import shutil
from pathlib import Path

print('Import done')

## 2. Définir les fonctions et la logique

Écrire les fonctions principales et la logique métier qui seront utilisées dans le notebook et le script Python.

In [ ]:
def normalize_tracking(track):
    if track is None:
        return ''
    return re.sub(r'[^A-Za-z0-9]', '', str(track).strip())


def read_dpd_csv(path):
    rows = []
    with open(path, encoding='latin-1', newline='') as f:
        reader = csv.DictReader(f, delimiter=';')
        for row in reader:
            rows.append(row)
    return rows


def filter_rows(rows):
    filtered = []
    for row in rows:
        typ = row.get('Type (Slave Export)', '').strip()
        if not typ.isdigit():
            continue
        if all(not str(v).strip() for v in row.values()):
            continue
        filtered.append(row)
    return filtered


def build_import_row(row):
    tracking = normalize_tracking(row.get('N° Colis') or row.get('Numero de suivi') or row.get('DPD ID'))
    return {
        'Transporteur': 'DPD',
        'Tracking': tracking,
        'Client': row.get('No de compte', '').strip(),
        'Pays': row.get('Destinataire pays', '').strip(),
        'Poids': row.get('Poids', '').strip(),
        'NbrColis': row.get('Nombre de colis', '').strip(),
        'Fret': row.get('Prix transport', '').strip(),
        'Gazole': row.get('Indexation gasoil', '').strip(),
    }


def process_dpd_files(csv_paths):
    rows = []
    for path in csv_paths:
        rows.extend(read_dpd_csv(path))
    rows = filter_rows(rows)
    return [build_import_row(r) for r in rows]

print('Functions defined')

## 3. Exécuter le code principal

Montrer un exemple d'utilisation des fonctions définies avec un petit jeu de données ou un cas d'usage simple.

In [ ]:
sample_rows = [
    {'Type (Slave Export)': '1', 'No de compte': '9748', 'N° Colis': '021-105124544 2',
     'Destinataire pays': 'FR', 'Poids': '10,5', 'Nombre de colis': '1',
     'Prix transport': '50,00', 'Indexation gasoil': '6,92'},
]
processed = [build_import_row(row) for row in sample_rows]
for row in processed:
    print(row)
print(f'{len(processed)} ligne(s) traitée(s)')

## 4. Générer et enregistrer le fichier Python

Exporter le code défini dans ce notebook vers un fichier Python `.py` pour réutilisation hors notebook.

In [ ]:
from nbconvert import PythonExporter

notebook_path = Path('c:/Users/Malak OUJDID/Downloads/Automatisation_Facture/notebooks/dpd_finalizer.ipynb')
output_path = Path('c:/Users/Malak OUJDID/Downloads/Automatisation_Facture/automatisation/finaliser_dpd.py')

with open(notebook_path, 'r', encoding='utf-8') as fh:
    nb_source = fh.read()

exporter = PythonExporter()
source, _ = exporter.from_filename(str(notebook_path))
with open(output_path, 'w', encoding='utf-8') as fh:
    fh.write(source)

print(f'Fichier Python généré : {output_path}')